# Calculating Lipinski Descriptors
Christopher Lipinski, a scientist at Pfizer, developed a set of rules for evaluating the drug-likeness of a compound - predicting if a molecule is likely to be orally active and bioavailable in humans. 
The rules is defined by the Absorption, Distribution, Metabolism and Excretion (ADME), also known as the pharmacokinetic profile. 
Lipinski's **Rule of Five** (Ro5)  

The Lipinski's Rules care strictly about the Absorption aspect of ADME (Absorption, Distribution, Metabolism, and Excretion):
When a pill is swallowed it must survive the stomach's acidity, remain adequately dissolved, and cross the non-polar cellular membranes of the gut lining to reach the bloodstream.
- Molecules that are too large (high molecular weight) move too slowly to easily penetrate membranes.
- Molecules with too many Hydrogen bond donors/acceptors bind too tightly to water and refuse to enter the non-polar membrane.
- Molecules with a Log P over 5 are highly water-insoluble (too greasy) and will not dissolve properly in the GI tract.

Compounds are generally considered to possesses favorable properties for oral administration if they do not violate more than one of the following criteria:

Molecular weight < 500 Dalton (g/mol)
Octanol-water partition coefficient (LogP) < 5 (a drug's octanol-water partition coefficient)
Hydrogen bond donors < 5 (total count of N-H and O-H bonds)
Hydrogen bond acceptors < 10 (total count of Nitrogen and Oxygen atoms)

> NOTES: 
- Not an absolute physical law - it is common for for classes of certain drugs to violate 2+ rules and still be viable for oral administration.
- Transporters: The rule assumes passive diffusion across membranes. Many large drugs trick the body into absorbing them by utilizing active transport proteins
- Specialized Drugs: Drugs targeting the central nervous system (CNS), peptide-based drugs, and large natural products often fall outside the Rule of Five.

In [1]:
import pandas as pd
import numpy as np
from openbabel import pybel
from IPython.display import SVG
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem.Draw import IPythonConsole
from chembl_webresource_client.new_client import new_client
IPythonConsole.ipython_useSVG = True

In [2]:
df = pd.read_csv('acetylcholinesterase_activity_03_classes_defined.csv')
df.head(3)

,molecule_chembl_id,canonical_smiles,standard_value,class
0,CHEMBL133897,CCOc1nn(-c2cccc(OCc3ccccc3)c2)c(=O)o1,750.0,active
1,CHEMBL336398,O=C(N1CCCCC1)n1nc(-c2ccc(Cl)cc2)nc1SCC1CC1,100.0,active
2,CHEMBL131588,CN(C(=O)n1nc(-c2ccc(Cl)cc2)nc1SCC(F)(F)F)c1ccccc1,50000.0,inactive


In [3]:
# Split by '.', find the longest string, and create a Series
smiles = (
    df['canonical_smiles']
    .astype(str)
    .str.split('.')
    .apply(lambda x: max(x, key=len))
)

# Set the name of the Series
smiles.name = 'canonical_smiles'


## Calculate Descriptors

In [6]:
def get_lipinski_descriptors(smiles, verbose=False) -> dict:
    """
    Calculate Lipinski's Rule of Five descriptors for a given SMILES string.
    Parameters:
    smiles (str): The SMILES string of the molecule.
    verbose (bool): Whether to print verbose output.

    Returns:
    dict: A dictionary containing the calculated descriptors.
    """
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        if verbose:
            print(f"Invalid SMILES: {smiles}")
        return None
    return {
        "desc_MolWeight": Descriptors.MolWt(mol),
        "desc_Molecular_LogP": Descriptors.MolLogP(mol),
        "desc+NumHDonors": Descriptors.NumHDonors(mol),
        "desc_HBA": Descriptors.NumHAcceptors(mol),
        "desc_TPSA": Descriptors.TPSA(mol),
        "desc_RotatableBonds": Descriptors.NumRotatableBonds(mol)
    }

In [7]:
df.head(3)

,molecule_chembl_id,canonical_smiles,standard_value,class
0,CHEMBL133897,CCOc1nn(-c2cccc(OCc3ccccc3)c2)c(=O)o1,750.0,active
1,CHEMBL336398,O=C(N1CCCCC1)n1nc(-c2ccc(Cl)cc2)nc1SCC1CC1,100.0,active
2,CHEMBL131588,CN(C(=O)n1nc(-c2ccc(Cl)cc2)nc1SCC(F)(F)F)c1ccccc1,50000.0,inactive


In [8]:
print(get_lipinski_descriptors("CCOc1nn(-c2cccc(OCc3ccccc3)c2)c(=O)o1"))

{'desc_MolWeight': 312.32500000000005, 'desc_Molecular_LogP': 2.8032000000000004, 'desc+NumHDonors': 0, 'desc_HBA': 5, 'desc_TPSA': 66.49000000000001, 'desc_RotatableBonds': 6}


In [15]:
df_lipinski = df.apply(lambda row: get_lipinski_descriptors(row['canonical_smiles']), axis=1)
print(df_lipinski)


0       {'desc_MolWeight': 312.32500000000005, 'desc_M...
1       {'desc_MolWeight': 376.9130000000002, 'desc_Mo...
2       {'desc_MolWeight': 426.8510000000001, 'desc_Mo...
3       {'desc_MolWeight': 404.8450000000001, 'desc_Mo...
4       {'desc_MolWeight': 346.33400000000006, 'desc_M...
                              ...                        
7170    {'desc_MolWeight': 449.2950000000001, 'desc_Mo...
7171    {'desc_MolWeight': 356.81600000000003, 'desc_M...
7172    {'desc_MolWeight': 414.46500000000026, 'desc_M...
7173    {'desc_MolWeight': 415.3240000000002, 'desc_Mo...
7174    {'desc_MolWeight': 601.2349999999999, 'desc_Mo...
Length: 7175, dtype: object


In [17]:
lipinski_colnames = ["MW", "LogP", "NumHDonors", "NumHAcceptors", "TPSA", "NumRotatableBonds"]
df_lipinski_2 = pd.DataFrame(df_lipinski.to_list())
df_lipinski_2.head()

,desc_MolWeight,desc_Molecular_LogP,desc+NumHDonors,desc_HBA,desc_TPSA,desc_RotatableBonds
0,312.325,2.8032,0,5,66.49,6
1,376.913,4.5546,0,4,51.02,4
2,426.851,5.3574,0,4,51.02,4
3,404.845,4.7069,0,4,51.02,3
4,346.334,3.0953,0,5,60.25,3


In [18]:
df_lipinski_2.columns = lipinski_colnames
df_lipinski_2.head(3)

,MW,LogP,NumHDonors,NumHAcceptors,TPSA,NumRotatableBonds
0,312.325,2.8032,0,5,66.49,6
1,376.913,4.5546,0,4,51.02,4
2,426.851,5.3574,0,4,51.02,4


In [19]:
df3 = pd.concat([df, df_lipinski_2], axis=1)
df3.head(3)

,molecule_chembl_id,canonical_smiles,standard_value,class,MW,LogP,NumHDonors,NumHAcceptors,TPSA,NumRotatableBonds
0,CHEMBL133897,CCOc1nn(-c2cccc(OCc3ccccc3)c2)c(=O)o1,750.0,active,312.325,2.8032,0,5,66.49,6
1,CHEMBL336398,O=C(N1CCCCC1)n1nc(-c2ccc(Cl)cc2)nc1SCC1CC1,100.0,active,376.913,4.5546,0,4,51.02,4
2,CHEMBL131588,CN(C(=O)n1nc(-c2ccc(Cl)cc2)nc1SCC(F)(F)F)c1ccccc1,50000.0,inactive,426.851,5.3574,0,4,51.02,4


## Convert IC50 to pIC50
Convert IC50 to negative logarithmic scale -log10(IC50) to improve uniformity of distribution.

- Take the IC50 values from the standard_value column and converts it from nM to M by multiplying the value by 10^-9
- Take the molar value and apply -log10
- Delete the standard_value column and create a new pIC50 column

NOTE: Vals > 100,000,000 will be reduced to 100,000,000 to prevent the -log value becoming negative.

In [20]:
# For each value in standard_value:
## Standardise the standard_value such that vals > 100,000,000 are replaced with 100,000,000
## Convert the standard_value from nM to M --> standard_value*10**-9
## Take the -log10 of the converted value --> -np.log10(standard_value*10**-9)
df3.standard_value.describe()

count    7.175000e+03
mean     1.225867e+05
std      2.131348e+06
min      0.000000e+00
25%      1.587450e+02
50%      2.410000e+03
75%      1.644000e+04
max      1.636817e+08
Name: standard_value, dtype: float64